# Build forcefield itp for polarizable glycans with compatibility with proteins and lipids

Note, this forcefield file is made for papetide-glycan interactions. Care must be taken to adapt this file for simualtions of lipids, ionic liquids, etc.


### 1. Load MARTINI 2.P parameters into a Dictionaty

#### 1.1 Read contents of martini_v2.P.itp into a list, and remove extra information

In [40]:
import pandas as pd 
import numpy as np
import re
import copy

In [41]:
with open("martini_v2.P_supp-material.itp") as f:
	martini_params = f.readlines()
# this = [x.strip() for x in content] 
print(martini_params[:10])

## Process params file
# First, remove all comment lines
martini_params = [m for m in martini_params if m[0] != ';']
# Second, remove all empty lines, i.e., 
# when stripped of white spaces and newline characters, lenght of the string ==0
martini_params = [m.strip() for m in martini_params if len(m.strip()) > 0]

['; FORCEFIELD V2.1P - POLARIZABLE WATER .. VERSION 14-04-2010 \n', ';\n', '; published online as supporting material for\n', '; \n', '; S.O. Yesylevskyy, L.V. Schafer, D. Sengupta, S.J. Marrink. \n', '; "Polarizable water model for the coarse-grained Martini force field."\n', '; PLoS Comp. Biol, 2010.  \n', '\n', '\n', '; This file contains all interaction parameters and topology for the \n']


#### 1.2 Separate each directive and its contents into the dictionary, 'directive_contents'

In [42]:
# Goal now is to separate each directive of the martini_params file
# First, we find the indices of the starting points of each directive.
# We can identify the starting points by looking for lines beginning with '['

directive_start_positions = [ [line, current_ind]          # each element contains directive_name the corresponding index
        for current_ind, line in enumerate(martini_params) # parse each line keeping track of the current index
        if re.search( '\[.+', line)]                       # .+ operator is equibalent to '*' in regex. '[' is a special character in regex, so \ should be used to find the literal '['

# The actual content of each directive starts after the '\[.+' line, and ends before the next '\[.+' line
# Now we will build a dictionary where directive_names are keys, and directive contents are values.
directive_contents = {}
i = 0
while i< len(directive_start_positions):
    directive_name, start_position = directive_start_positions[i]
    directive_name = directive_name.strip('[').strip(']').strip() # remove extra characters
    # print(directive_name)
    if i == len(directive_start_positions)-1: # special case of last element
        directive_contents[directive_name] = martini_params[start_position+1:] # load content till the end of the file
    else: 
        next_directive_start = directive_start_positions[i+1][1]
        # print(start_position, next_directive_start)
        directive_contents[directive_name] = martini_params[start_position+1:next_directive_start]
    i+=1
# 
directive_contents.keys()


dict_keys(['defaults', 'atomtypes', 'nonbond_params', 'moleculetype', 'atoms', 'constraints', 'angles', 'exclusions'])

In [43]:
"""
Make lists of regular, small, tiny, and special atoms
"""

small_beads = []
regular_beads = []
special_cases = []
for atom in directive_contents['atomtypes']:
    atom_name = atom.split()[0]
    if atom_name[0] == 'S':
        small_beads.append(atom_name)
    elif atom_name[0] in ['P', 'N', 'C', 'Q']:
        regular_beads.append(atom_name)
    else: # most likely if atom is D
        special_cases.append(atom_name)

# print(regular_beads, small_beads, special_cases)

tiny_beads = [s.replace('S', 'T') for s in small_beads]


#### 1.3 Clean the nonbond_params directive and store the interaction data in a dictionary

In [44]:

nb_param_copy = copy.deepcopy(directive_contents['nonbond_params'])

print(nb_param_copy[-10])

for l_index, l in enumerate(nb_param_copy):
    # l.split() splits l at whilespaces
    # print(len(l.split()))
    l_processed= l.split()

    # Now l_processed contains: atom_i, atom_j, funct, c6, c12, ... comments
    # we will keep the first 5 elements, and discard excess elements
    nb_param_copy[l_index] = l_processed[:5]
    # print(l)
print(nb_param_copy[-10])

D     C1      1       0.00000E-00     0.00000E-00 ; no LJ interaction
['D', 'C1', '1', '0.00000E-00', '0.00000E-00']


In [45]:
def check_dupliates(interaction_dictionary):
    int_dict = interaction_dictionary
    ## Check for duplicates in the int_dictionary
    sets = [ set(k) for k in int_dict.keys()]

    duplicates = []
    for key in int_dict.keys():
        key = set(key)
        y  = [i for i, e in enumerate(sets) if e == key]
        if len(y) > 1: 
            print(y)
            message_string = f'{str(key)} was duplicated at indices: {y}'
            duplicates.append(message_string)
    
    if len(duplicates)==0:
        duplicates.append('No duplicates found')
    print(duplicates)
    return duplicates
    


#### 1.4 Rewrite interactions in terms of sigma and epsilon. This is the final MARTINI parameter dictionary

In [46]:


# grab atom_i-->atom_j pairs
ij_pairs = [tuple(nbpar[0:2]) for nbpar in nb_param_copy] 
# get corresponding c6 and c12 terms
c6 = np.array([float(nbpar[3]) for nbpar in nb_param_copy])
c12 =  np.array([float(nbpar[4]) for nbpar in nb_param_copy])
# convert c6 c12 to sigmas and epsilons
sigmas_list = (c12/c6)**(1/6)
epsilons_list = (c6**2)/(4*c12)

# Interactions dictionary

martini_int_dict = {ij_pairs[i]: {
                'sigma': sigmas_list[i],
                'epsilon' : epsilons_list[i],
                } 
                for i in range(len(ij_pairs))
            }

check_dupliates(martini_int_dict)
len(martini_int_dict)

['No duplicates found']


/tmp/ipykernel_3037503/1614377944.py:7: RuntimeWarning: invalid value encountered in divide
  sigmas_list = (c12/c6)**(1/6)
/tmp/ipykernel_3037503/1614377944.py:8: RuntimeWarning: invalid value encountered in divide
  epsilons_list = (c6**2)/(4*c12)


740

### 2. Add new bead types to the interaction table
#### 2.1 Add tiny beads, these are based on MARTINI for DNA/RNA


In [47]:
## Make a working copy of martini_int_dict

int_dict = copy.deepcopy(martini_int_dict)

## Add tiny bead_types.

## write t--r terms
## these terms are identical to s--r terms
for t,s in zip(tiny_beads, small_beads):
    for j in regular_beads+special_cases:
        if (j,s) in int_dict:
            int_dict[(j,t)] = int_dict[(j,s)]
        elif (s,j) in int_dict:
            int_dict[(t,j)] = int_dict[(s,j)]

## write t--s terms
## these terms are identical to s--s terms
for t,s in zip(tiny_beads, small_beads):
    for j in small_beads:
        if (j,s) in int_dict:
            int_dict[(j,t)] = int_dict[(j,s)]
        elif (s,j) in int_dict:
            int_dict[(t,j)] = int_dict[(s,j)]

## write t--t terms
## these terms are identical to s--s terms, but sigma is 0.32 nm
for t,s in zip(tiny_beads, small_beads):
    for jt, js in zip(tiny_beads, small_beads):
        if (js, s) in int_dict:
            int_dict[(jt,t)] = {
                'sigma' : 0.32,
                'epsilon' : int_dict[(js,s)]['epsilon']
            }
        elif (s, js) in int_dict:
            int_dict[(t,jt)] = {
                'sigma' : 0.32,
                'epsilon' : int_dict[(s,js)]['epsilon']
            }

check_dupliates(int_dict)
int_dict_test =  copy.deepcopy(int_dict)
# # print(int_dict[('P5', 'SP5')], int_dict[('TP5', 'SP5')])
# print(int_dict[('TP1', 'TN0')])
# print(int_dict[('P1', 'TN0')])
# print(int_dict[('SP1', 'TN0')])

['No duplicates found']


#### 2.2 [Deprecated] For each interaction pair, calculate a size-dependent sigma
(See Pitfalls of the MARTINI Model, JCTC paper)
new_sigma = arithmetic average of the sigmas of each particle

In [48]:
## Now, let's edit all the sigmas

"""
2023-01-12 Update: We are not calculating size dependent sigma anymore
"""


# sigma_dict = {}
# for atom in regular_beads+small_beads+tiny_beads+special_cases:
#     if atom[0] == 'S':
#         sigma_dict[atom] = 0.43
#     elif atom[0] == 'T':
#         sigma_dict[atom] = 0.32
#     elif atom[0] == 'D':
#         sigma_dict[atom] = np.inf
#     else:
#         sigma_dict[atom] = 0.47

# # print(sigma_dict)
# new_sigmas = []
# for k in int_dict:
#     sigma_mean = (sigma_dict[k[0]]+sigma_dict[k[1]])/2
#     int_dict[k]['sigma'] = sigma_mean
#     new_sigmas.append(sigma_mean)

# print(np.unique(new_sigmas))

'\n2023-01-12 Update: We are not calculating size dependent sigma anymore\n'

#### 2.3 Defining new beads and their MARTINI-types they are based on
Parameters and names of new beads are derived from ProMPT

##### 2.3.1 Peptide information for determination of cation-pi interactions

In [49]:
## For now, set
# number of peptides,           
n_peptides          = 8
pep_seq = "AGCKNFFWKTFTSC" ; #"HSDGIFTDSYSRYRKQMAVKKYLAAVL"; 
#ALA-GLY-CYS-LYS-ASN-PHE-PHE-TRP-LYS-THR-PHE-THR-SER-CYS
aromatic_resids = [resid for resid, resname in enumerate(pep_seq) if resname in 'FYWH']
cationic_resids = [resid for resid, resname in enumerate(pep_seq) if resname in 'KR']

# number of aromatic residues,  
n_aromatic_residues = len(aromatic_resids)
n_cationic_residues = len(cationic_resids)

## New interactions will be stored in this list
new_interactions = set([])

In [50]:
n_aromatic_residues

4

##### 2.3.2 Add non-aromatic polarized beads

In [51]:
# These bead types are considered similar to the martini interactions

# similar = ['POL', 'C1', 'C2', 'C3', 'C4', 'C5', 
#             'SC1', 'SC2', 'SC3', 'SC4', 'SC5', 'Qa', 
#             'Q0', 'Qd', 'N0', 'Nd', 'Nda', 'SNd', 'SN0']

# beads to be polarized
to_be_polarized = ['P5', 'P4', 'P1', 'Na', 'Nda', 'N0', 'SP1', 'SNd', # from ProMPT
                        'TN0',                                  # for GAGs
                        ]

# Polarized beads of polar type are differentiated by a 'P' prefix
polarized_beads = ['P'+bead for bead in to_be_polarized]


##### 2.3.3 Adding polarized aromatic beads


In [52]:
# Add polarized bead types for apolar beads.
# aromatic beads are numbered based on no. of peptides and no. of aromatic residues
# this is done to account for cation-pi interactions later on.


polarized_beads += [f'ARP_r{resid+1}p{pepid+1}' for pepid in range(n_peptides) for resid in aromatic_resids]
to_be_polarized += [ 'SC1'      for i in range(n_aromatic_residues*n_peptides)] # aromatic beads are based on the MARTINI SC1 particle

##### 2.3.4 Adding non-polarized aromatic beads

In [53]:
# add regular aromatic beads
new_aromatic_beads = [f'AR_r{resid+1}p{pepid+1}' for pepid in range(n_peptides) for resid in aromatic_resids]

aromatic_reference_beads = ['SC1']*len(new_aromatic_beads) # aromatic beads are based on the MARTINI SC1 particle

print(new_aromatic_beads)
print(aromatic_reference_beads)
## LJ interactions of AR* and ARP* are like those of SC1


['AR_r6p1', 'AR_r7p1', 'AR_r8p1', 'AR_r11p1', 'AR_r6p2', 'AR_r7p2', 'AR_r8p2', 'AR_r11p2', 'AR_r6p3', 'AR_r7p3', 'AR_r8p3', 'AR_r11p3', 'AR_r6p4', 'AR_r7p4', 'AR_r8p4', 'AR_r11p4', 'AR_r6p5', 'AR_r7p5', 'AR_r8p5', 'AR_r11p5', 'AR_r6p6', 'AR_r7p6', 'AR_r8p6', 'AR_r11p6', 'AR_r6p7', 'AR_r7p7', 'AR_r8p7', 'AR_r11p7', 'AR_r6p8', 'AR_r7p8', 'AR_r8p8', 'AR_r11p8']
['SC1', 'SC1', 'SC1', 'SC1', 'SC1', 'SC1', 'SC1', 'SC1', 'SC1', 'SC1', 'SC1', 'SC1', 'SC1', 'SC1', 'SC1', 'SC1', 'SC1', 'SC1', 'SC1', 'SC1', 'SC1', 'SC1', 'SC1', 'SC1', 'SC1', 'SC1', 'SC1', 'SC1', 'SC1', 'SC1', 'SC1', 'SC1']


##### 2.3.5 Adding a special bead for proline sidechains

In [54]:
## Add APP5, a bead similar to PP5, but special for Proline
polarized_beads.append('APP5')
to_be_polarized.append('P5')
## Add PAC1, a bead similar to C1, but special for Proline
new_aromatic_beads += ['PAC1']
aromatic_reference_beads += ['C1']

In [55]:
"""
pi-pi and cation-pi interactions need AR, ARP, and Qd beads to be numbered.


## ARP is labelled as APR in the ProMPT scripts and ARP in the ProMPT paper
## Following assumes a system with 100 peptide of sequence KLVFFAE
n_aromatic_residues = 2 # There are two F's
n_peptides = 100        # There are 100 instances of the peptide

aro_hydrophobic = [f'AR{i+1}' for i in range(n_aromatic_residues*n_peptides)]
aro_polarizable = [f'ARP{i+1}' for i in range(n_aromatic_residues*n_peptides)] # I know ARP is a nonsensical name.
"""

"""
I am just going to define ARP and AR once. ARP Here, AR will be added with PAC1 (proline sidechain based on C1)
"""
# ARP and AR are both based on SC1
# ARP has internal dipoles to assist in stacking
# to_be_polarized.append('SC1') ; polarized_beads.append('ARP')



'\nI am just going to define ARP and AR once. ARP Here, AR will be added with PAC1 (proline sidechain based on C1)\n'

##### 2.3.5 Adding additional cationic beads

In [56]:
# What is AQd?
new_cationic_beads = [f'Qd_r{resid+1}p{pepid+1}' for pepid in range(n_peptides) for resid in cationic_resids]
cationic_reference_beads = ['Qd']*len(new_cationic_beads)
print(new_cationic_beads)

['Qd_r4p1', 'Qd_r9p1', 'Qd_r4p2', 'Qd_r9p2', 'Qd_r4p3', 'Qd_r9p3', 'Qd_r4p4', 'Qd_r9p4', 'Qd_r4p5', 'Qd_r9p5', 'Qd_r4p6', 'Qd_r9p6', 'Qd_r4p7', 'Qd_r9p7', 'Qd_r4p8', 'Qd_r9p8']


#### 2.4 Add placeholders for interaction terms for all new beads, and make corrections

In [57]:
reference_beads     = to_be_polarized+aromatic_reference_beads+cationic_reference_beads
newly_added_beads   = polarized_beads+new_aromatic_beads+new_cationic_beads

#### Add self-terms for new beads
for ref, new in zip(reference_beads, 
                      newly_added_beads):
    int_dict[(new, new)] = int_dict[(ref, ref)]
    new_interactions.add((new,new))

print(len(new_interactions))
check_dupliates(int_dict)
print(len(int_dict))
#########################################################################

### Add new_bead---new_bead interactions

test2 = []
i=0
while i < len(newly_added_beads)-1:
    new_i = newly_added_beads[i]
    ref_i = reference_beads[i]

    j = i+1 # skip self-interactions

    while j < len(newly_added_beads):
        new_j = newly_added_beads[j]
        ref_j = reference_beads[j]

        ref_pair = (ref_i, ref_j)
        new_pair = (new_i, new_j)
        if ref_pair in int_dict: # check if ref_pair is a key in int_dict
            int_dict[new_pair] = int_dict[ref_pair]
            new_interactions.add(new_pair)
            # test2.append(new_pair)
        elif ref_pair[::-1] in int_dict: # check if ref_pair is a key, but in reverse order, in int_dict
            int_dict[new_pair[::-1]] = int_dict[ref_pair[::-1]]
            new_interactions.add(new_pair[::-1])
            # test2.append(new_pair[::-1])
        j+=1
    i+=1


# #########################################################################

# #### Add cross-terms for new_beads--original_bead interactions
# #### values of cross terms are identical to those involving unpolarized beads
original_martini_beads = regular_beads+small_beads+tiny_beads+special_cases

i=0
while i< len(newly_added_beads):
    new_i = newly_added_beads[i]
    ref_i = reference_beads[i]
    for org_j in original_martini_beads:   
        ref_pair = (ref_i, org_j)
        new_pair = (new_i, org_j)
        if ref_pair in int_dict.keys():
            int_dict[new_pair] = int_dict[ref_pair]
            new_interactions.add(new_pair)
        elif ref_pair[::-1] in int_dict.keys():
            int_dict[new_pair[::-1]] = int_dict[ref_pair[::-1]]
            new_interactions.add(new_pair[::-1])
    i+=1

print(len(new_interactions))
check_dupliates(int_dict)
print(len(int_dict))

91
['No duplicates found']
1686
9282
['No duplicates found']
10877


In [58]:
def add_redundant_bead(interaction_data_frame, new_bead, reference_bead):

    """
    Usage:
    int_df = pd.DataFrame.from_dict(int_dict, orient='index')
    #### Add PAC1
    with_pac1 = add_redundant_bead(int_df, 'PAC1', 'C1')
    """
    int_df = interaction_data_frame.copy()
    new_bead=new_bead
    reference_bead=reference_bead

    for k0, k1 in int_df.index:
        
        # Add self-interaction
        if all([k0 == reference_bead, k1==reference_bead]):
            new_k0, new_k1 = new_bead, new_bead
            
        elif k0 ==reference_bead:
            new_k0, new_k1 = new_bead, k1
            
        elif k1 == reference_bead:
            new_k0, new_k1 = k0, new_bead
            
        else:
            continue
        
        ndf = pd.DataFrame(
            {
                "sigma": [int_df.loc[k0, k1]['sigma']],
                "epsilon": [int_df.loc[k0, k1]['epsilon']],
            }, 
            index=[(new_k0, new_k1)]
        )

        int_df = pd.concat([int_df, ndf])

    return int_df


##### 2.4.1 Water POL -- hydrophobic bead corrections


In [59]:
## Store int_dict as a dataframe. 
## Edits to epsilons will be made to the df alone
int_df = pd.DataFrame.from_dict(int_dict, orient='index')

In [60]:


"""
1. Change C1--POL, SC1--POL, PAC1--POL, AR*--POL, ARP*--POL interactions
This makes these hydophobic beads more averse to water
"""

# PAC1 and C1 are the same basically
# Naming of PAC1 differs to incorporate cation-pi and pi-pi interactions

int_df.loc['POL', 'C1']['epsilon'] = 1.0    
int_df.loc['POL', 'PAC1']['epsilon'] = 1.0
# I don't know why POL--SC1 is different from POL--AR
# However, SC1 is not used in the proteins, only AR is.
int_df.loc['POL', 'SC1']['epsilon'] = 1.0 
for ar_x in new_aromatic_beads:
    if ar_x[:3] == "AR_":
        int_df.loc['POL', ar_x]['epsilon'] = 1.5
for arp_x in polarized_beads:
    # Note this is corrected again, since ARP and POL are both polarized beads.
    if arp_x[:4] == 'ARP_':
        int_df.loc['POL', arp_x]['epsilon'] = 1.5  # Originally 1.9




##### 2.4.2 Aromatic (pi) -- Proline interactions

In [61]:
"""
2. Change AR*-APP5 interactions to 3.5 kJ/mol. Sort of simulates pi-proline interactions.
Makes interactions more attractive
## APP5--ARP will be ulimately corrected by the polar-polar interaction factor
"""

for ar_x in new_aromatic_beads:
    int_df.loc['APP5', ar_x]['epsilon'] = 3.5
for arp_x in polarized_beads:
    # Note this is corrected again, since ARP and APP5 are both polarized beads.
    if arp_x[:3] == 'ARP':
        int_df.loc['APP5', arp_x]['epsilon'] = 3.5  # Final value will be 3.5 - .58 ~ 2.9

# int_df.loc['APP5', 'AR']['epsilon'] = 3.5
# int_df.loc['APP5', 'ARP']['epsilon'] = 3.5  # Originally 1.9


##### 2.4.3 Polarized-Polarized corrections


I can directly borrow epsilons from martini 2.P. \
Then, correct their interactions by -.58 (a la ProMPT) for: 
>polarizable--polarizable interactions\
>polarizable--water interactions\
>polarizable--charged interactions

Then, calculate c6 and c12 terms based on arithmetic averages of sigmas. 

For apolar and aromatic interactions, set epsilons as done in proMPT. These corrections vary. Calculate c6 and c12 with appropriate average sigmas

In [62]:
"""
3. Make corrections to polarized-polarized interactions
"""

polarizable_correction_factor = 0.58 # epsilon correction in kJ/mol. Comes fro ProMPT, however, basis unknown.

## parse the list of all new interactions
## if the interaction involves 2 polarizable beads,
## Reduce their epsilon by the correction factor
# correction_factor = 0.58
list_of_polarized_polarized_interactions = {}
count=0
for (i, j) in list(new_interactions):
    if all(bead in polarized_beads for bead in [i, j]):
        
        old_eps = int_df.loc[i,j]['epsilon']
        # print('que', i, j)
        int_df.loc[i,j]['epsilon'] -= polarizable_correction_factor
        new_eps = int_df.loc[i,j]['epsilon']
        list_of_polarized_polarized_interactions[(i,j)] = [f"{old_eps:.3f}", f"{new_eps:.2f}"]
        count+=1

print(count)

verification_df = pd.DataFrame.from_dict(list_of_polarized_polarized_interactions ,orient='index')
verification_df.to_csv('polarized-polarized_interactions.csv')


903


##### 2.4.4 POL-polarized bead adjustments

In [63]:

"""
3. POL-polarized beads water adjustments
"""

list_of_polarized_POL_interactions = {}
polar_to_water_correction_factor = 0.58
count=0
for (i, j) in list(new_interactions):
    if 'POL' in [i,j]:
        if any(bead in polarized_beads for bead in [i, j]):
            old_eps = int_df.loc[i,j]['epsilon']
            # print('que', i, j)
            int_df.loc[i,j]['epsilon'] -= polar_to_water_correction_factor
            new_eps = int_df.loc[i,j]['epsilon']
            list_of_polarized_POL_interactions[(i,j)] = [f"{old_eps:.3f}", f"{new_eps:.2f}"]
            print(i,j)
            count+=1
print(count)
verification_df = pd.DataFrame.from_dict(list_of_polarized_POL_interactions ,orient='index')
verification_df.to_csv('polarized-POL_interactions.csv')

POL PN0
POL PP4
POL ARP_r7p6
POL ARP_r7p8
POL ARP_r6p6
POL PSP1
POL PNda
POL ARP_r6p3
POL ARP_r11p2
POL ARP_r8p3
POL PTN0
POL ARP_r7p3
POL ARP_r8p7
POL ARP_r8p8
POL ARP_r11p1
POL ARP_r7p4
POL ARP_r11p7
POL ARP_r7p2
POL ARP_r11p3
POL PSNd
POL ARP_r8p5
POL ARP_r11p6
POL ARP_r8p6
POL ARP_r6p5
POL APP5
POL PP1
POL ARP_r6p8
POL ARP_r6p7
POL ARP_r11p5
POL ARP_r6p2
POL ARP_r11p8
POL ARP_r6p1
POL PP5
POL ARP_r7p7
POL PNa
POL ARP_r8p1
POL ARP_r8p4
POL ARP_r11p4
POL ARP_r6p4
POL ARP_r7p5
POL ARP_r8p2
POL ARP_r7p1
42


##### 2.4.5 Charged-polarized interactions

In [64]:
"""
4. Charged-polarized edits
"""
charged_beads = [k for k in original_martini_beads if 'Q' in k]+new_cationic_beads
print(len(charged_beads))
# # print([k for k in polarized_beads if 'Q' in k])
q_correction_factor = 0.28

list_of_polarized_charged_interactions = {}

count=0
for (i, j) in list(new_interactions):
    ## check if one charged bead and one pol bead are present in (i,j)
    if all(
            [
                any(qbead in charged_beads for qbead in [i,j]),
                any(pbead in polarized_beads for pbead in [i, j])
                ]
            ):
            old_eps = int_df.loc[i,j]['epsilon']
            int_df.loc[i,j]['epsilon'] -= q_correction_factor
            new_eps = int_df.loc[i,j]['epsilon']
            list_of_polarized_charged_interactions[(i,j)] = [f"{old_eps:.3f}", f"{new_eps:.2f}"]
            # print(i,j)
            count+=1
print(count)
verification_df = pd.DataFrame.from_dict(list_of_polarized_charged_interactions ,orient='index')
verification_df.to_csv('polarized-charged_interactions.csv')


28
1176


In [65]:
# int_df.loc['POL', 'ARP_r4p1']
# # if ('PP5', 'Qd1') i
# ('PP5', 'Qd1') in int_df.index

##### 2.4.6 Cation-pi interactions

```
# this is a list of resid_pairs. Distance between aromatic/cationic residues are used to determine if there will be cation-pi interaction, or not.

Relevant = []
for i in range(len(Qd_resids)):
    for j in range(len(Ar_resids)):
        # If distance is 3 or more in the primary structure,
        if np.abs(Qd_resids[i] - Ar_resids[j]) > 2: 
            # Is this only intrapeptide? Appears so.
            S = [[i+(len(Qd_resids)*p), j+(len(Ar_resids)*p)] for p in range(numpeptides)]
            for t in S:
                Relevant.append( [t[0], t[1]] )

Relevant = np.array(Relevant)+1

epsilon = 3.0 #Change Here
sigma = 0.47
c6 = 4*epsilon*((sigma)**6)
c12 = 4*epsilon*((sigma)**12) 

for i in range(len(Relevant)):
    G3.edges['Qd'+str(Relevant[i][0]), 'AR'+str(Relevant[i][1])]['weight'] = epsilon
    G3.edges['Qd'+str(Relevant[i][0]), 'AR'+str(Relevant[i][1])]['another'] = np.array([c6, c12])
    
epsilon = 3.0-0.28 #Change Here
sigma = 0.47
c6 = 4*epsilon*((sigma)**6)
c12 = 4*epsilon*((sigma)**12) 

for i in range(len(Relevant)):
    G3.edges['Qd'+str(Relevant[i][0]), 'ARP'+str(Relevant[i][1])]['weight'] = epsilon
    G3.edges['Qd'+str(Relevant[i][0]), 'ARP'+str(Relevant[i][1])]['another'] = np.array([c6, c12])

tillnow = list(G3.nodes)
```

In [66]:
"""
5. Cation-pi edits ----------------------------------------------------
"""

## we need to exclude the beads of aminoacids less 3 amino acids away from cvation-pi interactions

print(pep_seq)
print(cationic_resids, aromatic_resids)

excluded_cation_pi_pairs = []
for i, cationic_ind in enumerate(cationic_resids):
    for j, aromatic_ind in enumerate(aromatic_resids):
        if np.abs(cationic_ind - aromatic_ind) <= 2:
            print(cationic_ind, aromatic_ind)
            for pep in range(n_peptides):
                Qd_AR = (f"Qd_r{cationic_ind+1}p{pep+1}", f"AR_r{aromatic_ind+1}p{pep+1}")
                excluded_cation_pi_pairs += [ Qd_AR, Qd_AR[::-1]] 
                Qd_ARP = (f"Qd_r{cationic_ind+1}p{pep+1}", f"ARP_r{aromatic_ind+1}p{pep+1}")
                excluded_cation_pi_pairs += [ Qd_ARP, Qd_ARP[::-1]] 

print(excluded_cation_pi_pairs)



# Select all beads of Qd* type: Qd, Qd_r0p0, etc
for i in [bead for bead in newly_added_beads if bead[:2] == 'Qd'] + ['Qd']:
    # Select all beads used in aromatic side chain rings: AR_* and ARP_*
    for j in [bead for bead in newly_added_beads if bead[:3] in ['ARP', 'AR_']]:
        
        if (i,j) in excluded_cation_pi_pairs:
            # if this condition is met, both for loops will be terminated
            print("excluded:", (i,j))
            continue
        
        if j[:3]=='ARP':
            eps = 3.0 - 0.28
        elif j[:3] == 'AR_':
            eps = 3.0
        else:
            # Do not touch any interactions that do not involve AR* and ARP*
            continue

        if (i,j) in int_df.index:
            int_df.loc[i,j]['epsilon'] = eps
            print(i,j)
        elif (j,i) in int_df.index:
            int_df.loc[j,i]['epsilon'] = eps
            print(j, i)

"""
----------------------------------------------------
"""


AGCKNFFWKTFTSC
[3, 8] [5, 6, 7, 10]
3 5
8 6
8 7
8 10
[('Qd_r4p1', 'AR_r6p1'), ('AR_r6p1', 'Qd_r4p1'), ('Qd_r4p1', 'ARP_r6p1'), ('ARP_r6p1', 'Qd_r4p1'), ('Qd_r4p2', 'AR_r6p2'), ('AR_r6p2', 'Qd_r4p2'), ('Qd_r4p2', 'ARP_r6p2'), ('ARP_r6p2', 'Qd_r4p2'), ('Qd_r4p3', 'AR_r6p3'), ('AR_r6p3', 'Qd_r4p3'), ('Qd_r4p3', 'ARP_r6p3'), ('ARP_r6p3', 'Qd_r4p3'), ('Qd_r4p4', 'AR_r6p4'), ('AR_r6p4', 'Qd_r4p4'), ('Qd_r4p4', 'ARP_r6p4'), ('ARP_r6p4', 'Qd_r4p4'), ('Qd_r4p5', 'AR_r6p5'), ('AR_r6p5', 'Qd_r4p5'), ('Qd_r4p5', 'ARP_r6p5'), ('ARP_r6p5', 'Qd_r4p5'), ('Qd_r4p6', 'AR_r6p6'), ('AR_r6p6', 'Qd_r4p6'), ('Qd_r4p6', 'ARP_r6p6'), ('ARP_r6p6', 'Qd_r4p6'), ('Qd_r4p7', 'AR_r6p7'), ('AR_r6p7', 'Qd_r4p7'), ('Qd_r4p7', 'ARP_r6p7'), ('ARP_r6p7', 'Qd_r4p7'), ('Qd_r4p8', 'AR_r6p8'), ('AR_r6p8', 'Qd_r4p8'), ('Qd_r4p8', 'ARP_r6p8'), ('ARP_r6p8', 'Qd_r4p8'), ('Qd_r9p1', 'AR_r7p1'), ('AR_r7p1', 'Qd_r9p1'), ('Qd_r9p1', 'ARP_r7p1'), ('ARP_r7p1', 'Qd_r9p1'), ('Qd_r9p2', 'AR_r7p2'), ('AR_r7p2', 'Qd_r9p2'), ('Qd_r9p2', 'ARP

'\n----------------------------------------------------\n'

In [67]:
# print(int_df.loc['AR_r4p1', 'Qd_r5p2'])
# print(int_df.loc['AR_r4p1', 'Qd_r5p1'])
# print(int_df.loc['AR_r4p1', 'Qd'])
# print(int_df.loc['SC1', 'Qd_r5p1'])
      

### 3. Corrections of epsilon values in accordance with ProMPT

#### From martini_v2.1-dna.itp:
| interaction | sigma |       epsilon|
|---|---|---|
|P5 - P5|     .47|         5.6|
|SP5 - SP5|   .43   |      4.2 |
|TP5 - TP5|   .32    |     4.2|



In [68]:
"""
Dummies
"""
# int_df = add_redundant_bead(int_df, 'D1', 'D')
# int_df = add_redundant_bead(int_df, 'D1A', 'D')
# int_df = add_redundant_bead(int_df, 'D2', 'D')
# int_df = add_redundant_bead(int_df, 'D2A', 'D')

## POL - dummy interactions are not present in martini_v2.P.
# Adding them here
pol_d = pd.DataFrame({"sigma": np.inf,
                            "epsilon": np.inf,
                            }, index=[('POL', 'D')])
int_df = pd.concat((int_df, pol_d))


## Add new_dummies---D
new_dummies = ['DT', 'D2', 'D2A'] # DT is dummy for tiny
for new_d in new_dummies:
    int_df = add_redundant_bead(int_df, new_d, 'D')
    new_d_interaction = pd.DataFrame({"sigma": np.inf,
                            "epsilon": np.inf,
                            }, index=[(new_d, 'D')])
    int_df = pd.concat((int_df,new_d_interaction))
    # print(int_df.loc['D2', 'D'])



In [69]:
# Sort int_df

y = int_df
# y.sort_values(by='level_0', ascending=False)
y.sort_index(inplace=True)

In [70]:
def convert_to_c6_c12(int_df):
    converted_df = pd.DataFrame({'c6': [None]*len(int_df),
                                    'c12': [None]*len(int_df),
                                    'sigma': [f'; {sig:.3f}' for sig in int_df['sigma']],
                                    'epsilon': [f'; {eps:.3f}' for eps in int_df['epsilon']],
                                    },
                                    index=int_df.index
                                    )

    for ind in converted_df.index:
        sig = int_df.loc[ind]['sigma']
        eps = int_df.loc[ind]['epsilon']
        c6 = 4*eps*(sig**6)
        c12 = 4*eps*(sig**12)
        sig = f'; {sig:.3f}'
        eps = f'; {eps:.3f}'
        converted_df.loc[ind] = [c6, c12, sig, eps]
        # sigma, epsilon = int_df.loc[k0, k1]
        # converted_df.loc[k0, k1]['c6'] = 4*epsilon*(sigma**6)
        # converted_df.loc[k0, k1]['c12'] = 4*epsilon*(sigma**12)

    # All D -- any bead interactions should be 0
    for ind in converted_df.index:
        if any([bead=='D' for bead in ind]):
            sig = '; no LJ interaction'
            eps = ''
            converted_df.loc[ind] = [0, 0, sig, eps]
    # Any interaction with the new dummies shoudld have a hardcore repuslion factor
    # new_dummies  = ['D1', 'D2', 'D1A', 'D2A']
    for ind in converted_df.index:
        if any([bead in new_dummies for bead in ind]):
            sig = '; sigma infinity'
            eps = ''
            converted_df.loc[ind] = [0, 4.5387e-10, sig, eps]
    ## new dummies --- standard Dummy (D) should also be repulsive
    for new_d in new_dummies:
        sig = '; sigma infinity'
        eps = ''
        converted_df.loc[new_d, 'D'] = [0, 4.5387e-10, sig, eps]

    return converted_df

x = convert_to_c6_c12(int_df)


## Assemble a forcefield file

In [71]:
"""
Write comments
"""

comment = [f"; Forcefield for self-assembly of heparin (any length and number)",
            f"; with {n_peptides} peptides of sequence {pep_seq}"]

"""
Write defaults directive
"""
defaults_directive = ['[ defaults ]', "1 1 no 0.0 0.0", '']

"""
Write atomtypes directive

classify all beads into following categories:
>>> martini_2P_beads        (from polarizable water paper)
>>> martini_2P_tiny_beads   (from nucleic acid extension)
>>> new_polarized_beads     
>>> new_dummies         
>>> new_non_polarized_beads
"""


## First, all the stanbdard martini_beads
martini_2P_beads = list(map(lambda x: x.split()[0],
                            directive_contents['atomtypes'])
                            )
martini_2P_tiny_beads = tiny_beads

## Now parse the newly made non_bonded parameters list to identify new beads
all_beads = []
for ind in x.index:
    all_beads += list(ind)
all_beads = pd.unique(all_beads)
new_beads = [bead for bead in all_beads 
                if bead not in martini_2P_beads+martini_2P_tiny_beads
                ]
## now to categorize the new_beadtypes
new_polarized_beads = polarized_beads # previously defined
new_dummies = new_dummies             # previously defined
new_non_polarized_beads = [bead for bead in new_beads 
                            if all([bead not in polarized_beads,
                                    bead not in new_dummies])
                            ]

"""
Now, create an atomtypes directive in GROMACS format
Mass of all standard/unpolarized regular beads = 72 amu
Mass of all standard/unpolarized small and tiny beads = 45 amu
In poalrized beads, total mass will be distributed across central, dummy+, and dummy- particles
i.e., PP*: 72/3; PS*: 45/3; PT*: 45/3 
"""
def atomtype_line(bead_name, mass):
    return f'{bead_name:<11} {mass} 0.00   A   0.0   0.0'
atomtypes_directive = ['[ atomtypes ]', '; name mass charge ptype c6 c12']
# add all martini_types
atomtypes_directive += ['; Beads from MARTINI 2.P nucleic acid extension']
for bead in martini_2P_beads+martini_2P_tiny_beads:
    if bead[0] in ['S', 'T']: # list of tiny and small polarized beads
        bead_mass = 45.0
    elif bead in ['POL', 'D']:
        bead_mass = 24.0
    else:
        bead_mass = 72.0
    atomtypes_directive += [atomtype_line(bead, bead_mass)]
# atomtypes_directive += [atomtype_line(bead,72.0) for bead in martini_2P_beads+martini_2P_tiny_beads]
# add all new polarized beads
atomtypes_directive += ['; Central particles of polarized beads with internal dipoles']
for bead in new_polarized_beads:
    if bead in ['PTN0', 'PSNd','APC1']: # list of tiny and small polarized beads
        bead_mass = 15.0
    elif bead[:3] == 'ARP': # list of tiny and small polarized beads
        bead_mass = 15.0
    else:
        bead_mass = 24.0
    atomtypes_directive += [atomtype_line(bead, bead_mass)]
## add new dummies
atomtypes_directive += ['; Dummy particles of polarized beads with internal dipoles']
for bead in new_dummies:
    if bead in ['DT', 'D2A']: # list of dummies for tiny and small polarized beads
        bead_mass = 15.0
    else:
        bead_mass = 24.0
    atomtypes_directive += [atomtype_line(bead, bead_mass)]
## add new non-polarized beads
atomtypes_directive += ['; Newly defined non-polarized beads']
for bead in new_non_polarized_beads:
    if bead[:2] == 'AR': # list of tiny and small non-polarized beads
        bead_mass = 15.0
    else:
        bead_mass = 24.0
    atomtypes_directive += [atomtype_line(bead, bead_mass)]
atomtypes_directive += ['']



In [72]:
"""
write nonbonded params directive
"""

nb_params_directive = ['[ nonbond_params ]', '; i j funct c6 c12; sigma epsilon']
def write_nbparam_line(ij_pair, c6_c12_df):
    i, j = ij_pair
    funct = 1
    c6, c12, sig, eps = c6_c12_df.loc[ij_pair]
    return f'{i:<11}\t{j:<11}\t {funct} {c6:.3E} {c12:.3E} {sig}{eps}'
for ind in x.index:
    # print(write_nbparam_line(ind, x))
    nb_params_directive += [write_nbparam_line(ind, x)]
nb_params_directive += ['']

In [73]:
"""
Print to file
"""

out_file = './ff_1hp_8SS14.itp'
with open(out_file, 'w') as f:
    outdata = '\n'.join(comment+defaults_directive+atomtypes_directive+nb_params_directive)
    f.write(outdata)

In [ ]:
# ##ISSUE:
# issue_int_dict = copy.deepcopy(int_dict)
# issue_df = pd.DataFrame.from_dict(int_dict, orient='index')

# print("dictionary intitial: ",issue_int_dict[('PP5', 'PP5')]['epsilon'])
# print("df intitial: ",issue_df.loc['PP5', 'PP5']['epsilon'])


# for k in issue_int_dict.keys():
#     k0, k1 = k
#     # print(df.loc[k0, k1]['epsilon'])
#     if (k0 in polarized_beads) and (k1 in polarized_beads): # if interaction is polarized
        
#         issue_df.loc[k0, k1]['epsilon'] -= 0.58
#         issue_int_dict[(k0, k1)]['epsilon'] -= 0.58

# print("dictionary final: ", issue_int_dict[('PP5', 'PP5')]['epsilon'])
# print("df final: ", issue_df.loc['PP5', 'PP5']['epsilon'])